# M2: Segmentación de Instancias de Vehículos (YOLO11-seg)
**IAM-811 Machine Learning y Aprendizaje Profundo** · Laboratorio 3: Sistema de Control de Peaje

Entrenamiento del modelo **M2 (Segmentación de Instancias)** para clasificar y segmentar las 5 categorías tarifarias del peaje:
- `moto` (Motocicleta)
- `auto` (Automóvil / Vagoneta / Camioneta particular)
- `minibus` (Minibús / Trufi / Van de transporte público)
- `camion` (Camión de carga / Tráiler)
- `bus` (Flota / Bus interdepartamental o urbano)

**Dataset:** Gestionado y etiquetado con polígonos de máscara en **Roboflow** (combinando imágenes de Roboflow Universe con capturas propias en peajes/vías locales).

In [ ]:
# 1. Instalación de Ultralytics y Roboflow
!nvidia-smi -L || echo "Active la GPU T4 en Entorno de ejecución > Cambiar tipo de entorno"
!pip install -q ultralytics==8.3.27 roboflow==1.1.48 opencv-python-headless pandas matplotlib

In [ ]:
# 2. Descarga del Dataset de Segmentación desde Roboflow
import os
from roboflow import Roboflow

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "TU_ROBOFLOW_API_KEY")
WORKSPACE = "peaje-iam811"
PROYECTO_M2 = "segmentacion-vehiculos-peaje"
VERSION_M2 = 1

if ROBOFLOW_API_KEY != "TU_ROBOFLOW_API_KEY":
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE).project(PROYECTO_M2)
    dataset = project.version(VERSION_M2).download("yolov11")
    DATA_YAML = os.path.join(dataset.location, "data.yaml")
    print("Dataset M2 descargado en:", DATA_YAML)
else:
    DATA_YAML = "coco128-seg.yaml"
    print("Reemplace ROBOFLOW_API_KEY con su clave real de Roboflow. Usando coco128-seg.yaml para prueba de pipeline.")

## 3. Entrenamiento de `yolo11n-seg.pt`
Entrenamos con aumentación de datos (mosaic, flip horizontal, variación HSV) para mejorar la generalización en condiciones de caseta (día, tarde, noche, sombras).

In [ ]:
from ultralytics import YOLO

modelo_seg = YOLO("yolo11n-seg.pt")

resultados = modelo_seg.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="runs_peaje",
    name="m2_vehiculos_seg",
    patience=15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0
)

## 4. Evaluación y Reporte de Métricas (mAP50 y mAP50-95 de Cajas y Máscaras)
Guarda las métricas requeridas para el Informe Técnico PDF (Sección 7.2 y 9).

In [ ]:
import shutil, json
import pandas as pd

metricas = modelo_seg.val()
resumen_m2 = {
    "modelo": "M2 - YOLO11n-seg (Segmentación de Instancias de Vehículos)",
    "box_mAP50": round(float(metricas.box.map50), 4),
    "box_mAP50_95": round(float(metricas.box.map), 4),
    "mask_mAP50": round(float(metricas.seg.map50), 4),
    "mask_mAP50_95": round(float(metricas.seg.map), 4),
    "precision_mask": round(float(metricas.seg.mp), 4),
    "recall_mask": round(float(metricas.seg.mr), 4)
}

print(json.dumps(resumen_m2, indent=2, ensure_ascii=False))
with open("metricas_M2_segmentacion.json", "w") as f:
    json.dump(resumen_m2, f, indent=2)

mejor_pt = "runs_peaje/m2_vehiculos_seg/weights/best.pt"
if os.path.exists(mejor_pt):
    shutil.copy(mejor_pt, "m2_vehiculos_seg.pt")
    print("Pesos exportados exitosamente a: m2_vehiculos_seg.pt")